# Live demo in the lab

Ask the system a question and watch the whole pipeline: rate limit, input guardrail, memory, hybrid retrieval, prompt, qwen3:8b on the A100, output guardrail. This is the real system; nothing is replayed. Run the cells in order inside a booked lab session (Ollama must be running: `bash scripts/lab_start.sh qwen3:8b`).

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.home() / 'multimodal-rag-group22'
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
from rag.demo import STYLES, build_system, demo_options, to_markdown
SYSTEM = build_system(os.path.expanduser('~/rag_index'), embedder='st', llm='ollama', model='qwen3:8b')
print('ready:', len(SYSTEM.index), 'chunks | retrieval:', SYSTEM.mode, '| model:', SYSTEM.llm_label)

In [ ]:
import ipywidgets as w
from IPython.display import Markdown, clear_output, display

example = w.Dropdown(options=[('', '')] + demo_options(ROOT), description='Example', layout=w.Layout(width='95%'))
question = w.Textarea(placeholder='Type a question about the RBI documents', layout=w.Layout(width='95%', height='70px'))
style = w.Dropdown(options=STYLES, value='create', description='Prompt style')
ask = w.Button(description='Ask', button_style='primary')
forget = w.Button(description='Forget the conversation')
out = w.Output()

def pick(change):
    question.value = change['new']
example.observe(pick, names='value')

def on_ask(_):
    with out:
        clear_output()
        if not question.value.strip():
            print('Type a question first.')
            return
        display(Markdown(to_markdown(SYSTEM.ask(question.value, variant=style.value))))

def on_forget(_):
    SYSTEM.reset()
    with out:
        clear_output()
        print('Conversation memory cleared.')

ask.on_click(on_ask)
forget.on_click(on_forget)
display(w.VBox([example, question, w.HBox([style, ask, forget]), out]))

The first question after a restart takes about 30 seconds while the model loads; the following ones take about 1 second. The Example list holds the 8 demo steps, all 30 golden questions (label shows the gold answer and page) and the guardrail test cases (label shows the expected verdict). For the video, use the 8 demo steps in order: a text question, a table question, a follow-up that needs memory, a chart question, an injection, an off-topic question, personal data, and a question with no evidence.